In [1]:
from tara_preprocessing import get_just_ecog_data,get_electrode_normalized_loc,car
from noah_production_funcs import single_patient_prediction_pure,create_lapaican_rbf
from tara_preprocessing import remove_duplicates, hold_out, preprocessing,apply_car_function,clip_time_series
from tara_preprocessing import make_patient_correlation_matrix
from noah_production_funcs import create_u
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import geoopt

ModuleNotFoundError: No module named 'tara_preprocessing'

In [ ]:
data_root = Path("/Users/noahwanless/Desktop/Spring2026/M467/faces_basic/data")
registered_dir = Path("../SuperEeg-M467-project/registered_outputs")
ecogs = get_just_ecog_data(registered_dir,data_root)
xyz = get_electrode_normalized_loc(registered_dir)
print('Downloaded data')
ecogs = clip_time_series(ecogs)
print("Time series clipped")
ecogs_no_dups,xyz_no_dups = remove_duplicates(ecogs,xyz)
print('Removed duplicate electrodes')
xyz_clea, cleane = preprocessing(ecogs_no_dups,xyz_no_dups,notch_size=.05)
print("Done Preprocessing")
cleaned_f,xyz_f,fake_pat_beginning = hold_out(xyz_clea,cleane,0,[40,41])
cleaned_f = apply_car_function(cleaned_f,0)
print("Done holding out electrodes")
patient_corr_mat = make_patient_correlation_matrix(xyz_f,cleaned_f)
print('Got Correlation Matrices, done!')


In [ ]:
import cvxpy as cp


def object_func_2(K,C,L_sqrt,lamb,patient_node_num,num_pat):
    sum = 0
    iter = 0
    for i in range(num_pat):
        c = cp.Parameter((C[i].shape[0],C[i].shape[0])) #each patient correlation matrix
        c.value = C[i].to_numpy()
        num_nodes = patient_node_num[i]
        k = K[iter:iter+num_nodes,iter:iter+num_nodes] #all columns of rows and colums iter+num of nodes + 1 (this gets just that patients correlation matrix in K)
        diff = (k-c)
        sum = sum + (cp.norm(diff,p='fro'))**2
        iter = iter + num_nodes
    sum = sum + lamb*cp.trace(L_sqrt@K) #quad_form
    return sum

from scipy.linalg import sqrtm


def optmize_k(num_elec,object_func,C,L,lamb,patient_node_num,num_pat):
    #L = cp.Parameter(L)
    L_sqrt = np.real(sqrtm(L))
    L_sqrt_cp = cp.Parameter((L_sqrt.shape[0], L_sqrt.shape[1]),PSD=True)
    L_sqrt_cp.value = L_sqrt

    lamb_cp = cp.Parameter()
    lamb_cp.value= lamb
    K = cp.Variable((num_elec,num_elec))
    objective = cp.Minimize(object_func(K,C,L_sqrt_cp,lamb_cp,patient_node_num,num_pat))
    constraints = [K>>0, cp.diag(K) == 1]
    problem = cp.Problem(objective, constraints)
    print("problem is DCP:", problem.is_dcp())
    problem.solve(verbose=True)
    return K

In [ ]:
L = create_lapaican_rbf(xyz_f,20)
num_pat = len(cleaned_f)
patient_node_num = []
for pat in cleaned_f:
    patient_node_num.append(pat.shape[1])
K = optmize_k(xyz_f.shape[0],object_func_2,patient_corr_mat,L,0.01,patient_node_num,num_pat)